## Convert PDF to Image & Crop

This is a helper tool used to convert handwritten notes from PDF to images and crop such that excessive white parts (i.e., the color appearing most frequently) are removed.

## Libraries

In [1]:
from pdf2image import convert_from_path

In [2]:
from PIL import Image
import numpy as np

## Code

In [ ]:
poppler_path = None  # Set to your local poppler bin path if needed, e.g. r'C:\...\poppler\Library\bin'
pages = convert_from_path('images/class_diagram3.pdf', 140, poppler_path=poppler_path) # Specify path to your PDF file

In [10]:
def crop_dominant_color_and_resize(pil_image, max_size=1024, tolerance=10, padding=10):
    img = pil_image.convert("RGB")
    arr = np.array(img)

    # --- 1. Find dominant color (most frequent pixel) ---
    pixels = arr.reshape(-1, 3)
    rounded = (pixels // 10) * 10
    colors, counts = np.unique(rounded, axis=0, return_counts=True)
    dominant_color = colors[np.argmax(counts)]

    # --- 2. Build a mask of non-background pixels ---
    diff = np.abs(arr.astype(int) - dominant_color.astype(int))
    is_background = np.all(diff <= tolerance, axis=2)
    is_foreground = ~is_background

    # --- 3. Compute bounding box of foreground + padding ---
    rows = np.any(is_foreground, axis=1)
    cols = np.any(is_foreground, axis=0)

    if not rows.any() or not cols.any():
        cropped = img
    else:
        h, w = arr.shape[:2]
        row_min, row_max = np.where(rows)[0][[0, -1]]
        col_min, col_max = np.where(cols)[0][[0, -1]]

        # Clamp padding to image bounds
        x0 = max(col_min - padding, 0)
        y0 = max(row_min - padding, 0)
        x1 = min(col_max + 1 + padding, w)
        y1 = min(row_max + 1 + padding, h)

        cropped = img.crop((x0, y0, x1, y1))

    # --- 4. Resize so largest dimension <= max_size ---
    w, h = cropped.size
    scale = min(max_size / w, max_size / h, 1.0)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cropped.resize((new_w, new_h), Image.LANCZOS)

    return resized

In [ ]:
processed_pages = [crop_dominant_color_and_resize(page) for page in pages]

# Save or inspect
for i, img in enumerate(processed_pages):
    print(f"Page {i+1}: {img.size}")
    img.save(f"images/class_diagram3/page_{i+1}_processed.jpg")